# 06 — RecBole: SASRec (sequential neural baseline)

Trains **SASRec** (transformer, sequential) and scores it through our shared
`evaluate_predictions` — the SAME harness/metrics as the classical, hybrid and LLM
methods, so the number drops straight into the UC1 comparison table.

**Requires a CUDA GPU** (Colab T4 or the Windows box — not Apple MPS). Install:
```
pip install recbole    # may pin numpy/pandas; use a fresh env if it clashes
```

**Split:** leave-one-out by time, matching `leave_last_n_out(sample, n=1)` — each
user's last interaction is the test target, the second-to-last is validation.

**Scale caveat:** ~12.6M interactions / 50k users. Fine on a T4; the 4 GB Quadro may
OOM → subsample users if needed. RecBole is **GPU (PyTorch)**, not TPU.

In [ ]:
# --- Make the FULL book_recsys package importable (Kaggle/Colab) ---
# Guard on book_recsys.DATA (not just book_recsys): a stale/partial top-level package can be
# importable while the subpackages are missing — that's the "No module named 'book_recsys.data'"
# error. Find the repo among mounted inputs, else clone it; put its root FIRST on sys.path and
# drop any partial already imported so the complete package resolves.
import sys, os, glob, subprocess
try:
    import book_recsys.data  # noqa: F401
except ModuleNotFoundError:
    hits = glob.glob('/kaggle/input/**/book_recsys/data/__init__.py', recursive=True)
    root = (os.path.dirname(os.path.dirname(os.path.dirname(hits[0]))) if hits
            else '/kaggle/working/book_recsys_src')
    if not hits and not os.path.exists(root):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/MayaDeneva/book_recsys', root], check=True)
    sys.path.insert(0, root)
    for m in [k for k in list(sys.modules) if k == 'book_recsys' or k.startswith('book_recsys.')]:
        del sys.modules[m]
    import book_recsys.data  # noqa: F401
import book_recsys
print('book_recsys at', book_recsys.__file__)

In [ ]:
# Keep Kaggle/Colab's native NumPy 2.x — do NOT downgrade numpy. Downgrading breaks the
# binary ABI of every preinstalled package built against NumPy 2 ("numpy.dtype size
# changed"). Just install RecBole; the next cell patches the 3 NumPy-2.0-removed aliases
# its compat shim needs. (If you previously downgraded numpy: Run -> Factory reset first.)
# kmeans-pytorch is an optional RecBole dep that `pip install recbole` doesn't pull in.
!pip install -q recbole kmeans-pytorch

In [ ]:
import os
import numpy as np
import pandas as pd

# RecBole's compatibility_settings() references aliases NumPy 2.0 removed (np.float_,
# np.complex_, np.unicode_), crashing on Kaggle/Colab's native NumPy 2.x. Re-add them so
# RecBole runs WITHOUT downgrading numpy (a downgrade ABI-breaks pandas/scipy). RecBole
# then restores np.float / np.int / etc. itself for the rest of its code.
for _old, _new in [("float_", "float64"), ("complex_", "complex128"), ("unicode_", "str_")]:
    if not hasattr(np, _old):
        setattr(np, _old, getattr(np, _new))

from book_recsys.data.splits import leave_last_n_out
from book_recsys.data.schema import USER, BOOK
from book_recsys.eval.harness import build_relevance, evaluate_predictions
from book_recsys.recbole_adapter.atomic import write_inter_file
from book_recsys.recbole_adapter.export import recbole_predictions

In [ ]:
import glob

def find_data(name):
    """Locate a data file locally (artifacts/) or on Kaggle (/kaggle/input/<dataset>/)."""
    for path in [f"artifacts/{name}", *glob.glob(f"/kaggle/input/**/{name}", recursive=True)]:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"{name} not in artifacts/ or /kaggle/input/** — attach the dataset")

sample = pd.read_parquet(find_data("sample.parquet"))
print(f"loaded {sample['user_id'].nunique()} users, {len(sample):,} interactions")

# --- Memory control for SASRec sequence augmentation (one sub-sequence per interaction) ---
# 1) Subsample users. A RANDOM subset is an unbiased estimate of the SAME NDCG@10, so it
#    stays comparable to the other methods. With the history cap below you can afford more.
N_USERS = 20000
keep = sample["user_id"].drop_duplicates().sample(N_USERS, random_state=42)
sample = sample[sample["user_id"].isin(keep)]

# 2) Cap each user's history to the most recent MAX_HIST interactions. Power users (max
#    ~67k) otherwise generate ~67k sub-sequences each, dominating RAM and the training
#    signal. SASRec only attends to the last MAX_ITEM_LIST_LENGTH (50) items anyway, so
#    trimming older history is principled, not just a hack. tail() keeps the recent items,
#    so each user's held-out LAST interaction is preserved.
MAX_HIST = 100
sample = (sample.sort_values(["user_id", "timestamp"])
                .groupby("user_id", sort=False).tail(MAX_HIST).reset_index(drop=True))
print(f"after subsample + history cap: {sample['user_id'].nunique()} users, "
      f"{len(sample):,} interactions")

# Our shared holdout = each user's last interaction by time. Classical/LLM methods
# are scored against this same `relevance`, so the SASRec number is comparable.
train, holdout = leave_last_n_out(sample, n=1)
relevance = build_relevance(holdout)

DATASET = "goodreads"
ds_dir = f"recbole_data/{DATASET}"
os.makedirs(ds_dir, exist_ok=True)
# Write the FULL (subsampled, capped) data; RecBole reproduces leave-one-out itself (LS
# split in the next cell), so its internal test == our `holdout`. Do NOT pre-split here.
write_inter_file(sample, f"{ds_dir}/{DATASET}.inter")

In [ ]:
import os
# reduce CUDA fragmentation (set before torch/recbole import; restart kernel if already imported)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# --- 1. Config ---
import json
from logging import getLogger

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer, init_logger, init_seed
from recbole.utils.case_study import full_sort_topk

MODEL = "SASRec"
config = Config(model=MODEL, dataset=DATASET, config_dict={
    "data_path": "recbole_data", "dataset": DATASET,
    "USER_ID_FIELD": "user_id", "ITEM_ID_FIELD": "item_id",
    "RATING_FIELD": "rating", "TIME_FIELD": "timestamp",
    "load_col": {"inter": ["user_id", "item_id", "rating", "timestamp"]},
    # Leave-one-out by time == our leave_last_n_out(n=1): last item -> test,
    # second-to-last -> validation. `mode: full` ranks against the WHOLE catalog
    # (matches our harness — NOT sampled negatives, which would inflate metrics).
    "eval_args": {"split": {"LS": "valid_and_test"}, "order": "TO",
                  "group_by": "user", "mode": "full"},
    # Memory lever: shorter sequences = fewer/smaller padded rows. Drop to 20 if RAM-tight.
    "MAX_ITEM_LIST_LENGTH": 50,
    # SASRec uses CE (softmax over all items) loss -> negative sampling must be OFF.
    "train_neg_sample_args": None,
    "epochs": 50, "topk": 10,
    # Speed knobs for the Kaggle/Colab GPU run:
    # SASRec full-softmax CE allocates a (batch x n_items=~468k) logits tensor + its
    # gradient: batch 4096 ~= 7.6GB *2 -> OOMs a 16GB T4. 512 ~= 0.95GB. Lower if still OOM.
    "train_batch_size": 512,
    "eval_batch_size": 256,       # full-sort eval over ~468k items is also memory-heavy
    "eval_step": 5,               # validate every 5 epochs — full-sort over 468k items is slow
    "stopping_step": 3,           # early-stop patience, in eval steps
    "metrics": ["NDCG", "Recall", "MRR"], "valid_metric": "NDCG@10",
    # Cache the processed dataset/dataloaders -> re-runs reload in seconds instead of
    # rebuilding (NB: doesn't lower the FIRST build's peak RAM — the user subsample does).
    "save_dataset": True, "save_dataloaders": True,
    # Checkpoints + logging (a dropped Kaggle session loses an untracked run):
    "checkpoint_dir": "saved",    # best model written here; under cwd (=/kaggle/working/...)
    "show_progress": True,        # per-epoch tqdm bars
    "seed": 42, "reproducibility": True,
})
init_seed(config["seed"], config["reproducibility"])
init_logger(config)               # streams per-epoch loss/valid; writes log/<run>.log
logger = getLogger()

In [ ]:
# --- 2. Build dataset + leave-one-out split (logs user/item/interaction counts) ---
dataset = create_dataset(config)
logger.info(dataset)
train_data, valid_data, test_data = data_preparation(config, dataset)

In [ ]:
# --- 3. Train (OR skip this and use the 'Load from checkpoint' cell below) ---
model = get_model(config["model"])(config, train_data.dataset).to(config["device"])
trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)
best_score, best_result = trainer.fit(train_data, valid_data, saved=True, show_progress=True)
logger.info("[%s] best valid %s -> checkpoint %s", MODEL, best_result, trainer.saved_model_file)

### (Alternative to training) Load a trained SASRec from its checkpoint
Use this **instead of cell 3 above** if the session died and you have `SASRec.pth`. Run the
scaffolding cells first (install → imports → data+`.inter` → config → build dataset), then this
— it skips `trainer.fit`. Attach `SASRec.pth` (in `artifacts/` or as a Kaggle input). The
rebuilt dataset must match the trained one — it does, since the subsample is deterministic
(same `sample.parquet`, `N_USERS`, `MAX_HIST`, seed).

In [ ]:
import torch
from recbole.utils import get_model
model = get_model(config['model'])(config, train_data.dataset).to(config['device'])
ckpt = torch.load(find_data('SASRec.pth'), map_location=config['device'], weights_only=False)
model.load_state_dict(ckpt['state_dict'])
if ckpt.get('other_parameter'):
    model.load_other_parameter(ckpt['other_parameter'])
model.eval()
print('loaded SASRec:', ckpt.get('epoch'), 'epochs | best valid', ckpt.get('best_valid_score'))

In [ ]:
# --- Grab the trained model NOW. /kaggle/working is wiped when the interactive session
# ends unless you Save Version, so copy the checkpoint to a stable name + download it. ---
import shutil
from IPython.display import FileLink
os.makedirs('artifacts', exist_ok=True)
ckpt = f'artifacts/{MODEL}.pth'
shutil.copy(trainer.saved_model_file, ckpt)
print(f'best checkpoint {trainer.saved_model_file} -> {ckpt} ({os.path.getsize(ckpt)/1e6:.1f} MB)')
display(FileLink(ckpt))   # <- click to download to your machine before the session ends

In [ ]:
# --- 4. Export top-K per test user, BATCHED over users to avoid OOM. full_sort scores
# every user against ALL ~468k items, so all ~20k users in one call would need ~37 GB.
# 512 users/call ~= 0.96 GB. Move each result to CPU + empty_cache between batches. ---
import torch
USERS_PER_BATCH = 512
internal_users = list(range(1, dataset.user_num))  # skip [PAD]=0
chunks = []
for s in range(0, len(internal_users), USERS_PER_BATCH):
    batch_users = internal_users[s:s + USERS_PER_BATCH]
    _, iid = full_sort_topk(batch_users, model, test_data, k=10, device=config['device'])
    chunks.append(iid.cpu())
    torch.cuda.empty_cache()
topk_iid = torch.cat(chunks, dim=0)
uid2token = {i: dataset.id2token(dataset.uid_field, i) for i in internal_users}
iid2token = {int(i): dataset.id2token(dataset.iid_field, int(i))
             for i in topk_iid.reshape(-1).unique().tolist()}
preds = recbole_predictions(internal_users, topk_iid.tolist(), uid2token, iid2token)
print(f'exported top-10 for {len(preds):,} users')

In [ ]:
# --- 5. Sanity-check the riskiest seam: internal ids map back to ids we recognize ---
u0 = internal_users[0]
print("RecBole user", u0, "-> our user_id", uid2token[u0])
print("its top-10 book_ids:", preds[uid2token[u0]][:10])
print("# users predicted:", len(preds), "| # users in relevance:", len(relevance))

In [ ]:
# --- 6. Score through our shared harness + persist ---
results = {MODEL: evaluate_predictions(preds, relevance, k=10)}
with open("sasrec_results.json", "w") as fh:
    json.dump(results, fh, indent=2)
pd.DataFrame(results).T

In [ ]:
# --- Persist predictions (what the shared harness scores) + download links. preds are
# tiny and re-scorable without the GPU; the .pth is only needed to re-export or serve. ---
from IPython.display import FileLink
with open(f'artifacts/{MODEL}_preds.json', 'w') as fh:
    json.dump(preds, fh)
print('Download these now, OR Save Version to persist /kaggle/working as Output:')
for f in [f'artifacts/{MODEL}.pth', f'artifacts/{MODEL}_preds.json', 'sasrec_results.json']:
    display(FileLink(f))

## SASRec vs baselines — popularity-weighted sampled negatives (08's fair protocol)
Sampled-negatives needs scores over a *candidate set*, so it runs here (model required), not
from the exported top-10. For each test user we sample 100 **popularity-weighted** negatives
(our tested `sample_negatives`), then score the **same** candidates with SASRec (`full_sort_scores`)
**and** the in-process methods from `07`'s `models.joblib` — identical users + candidates = a fair
head-to-head against the hybrid. Needs `models.joblib` (work-level) attached.

In [ ]:
import json, statistics, joblib
from recbole.utils.case_study import full_sort_scores
from book_recsys.eval.harness import build_user_histories
from book_recsys.data.negatives import build_cdf, sample_negatives
from book_recsys.eval.metrics import recall_at_k, ndcg_at_k, mrr

histories = build_user_histories(train)
counts = train[BOOK].value_counts()
pool = counts.index.to_numpy()            # candidate book_id pool
cdf = build_cdf(counts.to_numpy())        # popularity-weighted negatives
rng, N_NEG, K = np.random.default_rng(0), 100, 10

models = joblib.load(find_data('models.joblib'))   # work-level zoo from 07_models
baselines = {n: models[n] for n in ['hybrid_cf_content', 'svd', 'content_emb'] if n in models}

uid_field, iid_field = dataset.uid_field, dataset.iid_field
known_users = set(dataset.field2id_token[uid_field])
def to_iid(tok):
    try: return dataset.token2id(iid_field, str(tok))
    except ValueError: return None

eval_users = [u for u in relevance if str(u) in known_users]
agg = {m: {'recall@10': [], 'ndcg@10': [], 'mrr': []} for m in ['SASRec', *baselines]}
BATCH = 256
for s in range(0, len(eval_users), BATCH):
    batch = eval_users[s:s + BATCH]
    uids = [dataset.token2id(uid_field, str(u)) for u in batch]
    sas = full_sort_scores(uids, model, test_data, device=config['device']).cpu().numpy()
    for row, u in zip(sas, batch):
        positive = next(iter(relevance[u]))
        seen = set(histories.get(u, [])) | {positive}
        cands = [positive] + sample_negatives(pool, seen, N_NEG, rng, cdf)
        scored = {'SASRec': [row[to_iid(c)] if to_iid(c) is not None else -np.inf for c in cands]}
        for name, mdl in baselines.items():
            scored[name] = mdl.score_items(histories.get(u, []), cands)
        for name, sc in scored.items():
            order = [cands[j] for j in np.argsort(sc)[::-1]]
            agg[name]['recall@10'].append(recall_at_k(order, {positive}, K))
            agg[name]['ndcg@10'].append(ndcg_at_k(order, {positive}, K))
            agg[name]['mrr'].append(mrr(order, {positive}))
res = {m: {k: statistics.fmean(v) for k, v in d.items()} for m, d in agg.items()}
print(f'popularity-weighted sampled-neg on {len(eval_users)} users:')
display(pd.DataFrame(res).T.sort_values('ndcg@10', ascending=False).round(4))
json.dump(res, open('artifacts/sasrec_popneg.json', 'w'))
print('saved artifacts/sasrec_popneg.json')

## Results, checkpoints & logging

The DataFrame above (recall@10 / ndcg@10 / mrr) feeds `reports/model_report.md`,
in the same UC1 table as svd / hybrid / content / llm. It's also written to
`sasrec_results.json`.

- **Checkpoints:** `saved=True` writes the best model (by NDCG@10) to `saved/` as it
  improves — survives a crash/timeout. Resume a dropped run with
  `trainer.resume_checkpoint("saved/<file>.pth")` before `trainer.fit`.
- **Progress:** `show_progress=True` shows per-epoch tqdm bars; `init_logger` streams
  per-epoch train-loss / valid-NDCG to stdout and writes `log/<run>.log`.
- **Kaggle persistence:** files under `/kaggle/working` (`saved/`, `sasrec_results.json`,
  `log/`) only persist when you **Save Version (Commit)**. For a multi-hour run use
  *Save & Run All (Commit)* so it trains headless and keeps the output even if your
  browser disconnects.

**Aligned:** RecBole's internal leave-one-out test == our `holdout`, and `mode: "full"`
ranks the whole catalog — SASRec is measured on the exact ruler as every other method.
**Sanity-check once:** confirm `uid2token` / `iid2token` map RecBole ids back to
user/book ids you recognize (the riskiest seam; adapter unit tests cover the mapping).

## Live SASRec serving — load model + item map + recommend (self-contained)

Everything needed to serve SASRec live, grouped at the bottom. `load_data_and_model` rebuilds config + dataset + model from `SASRec.pth` in one call (needs the dataset atomic files present, as they are right after training). The `book_id ↔ internal item id` map is exported **from the loaded model's dataset** (checkpoint-specific; not in UCSD data), and `sasrec_recommend()` is the reference impl for the `book_recsys` `SasrecRecommender`. No user map — SASRec is item-sequence only.

In [ ]:
# --- Load the trained model (CPU or GPU) + export the item map (from the loaded dataset) ---
import json, functools, torch
from recbole.data import create_dataset, data_preparation
from recbole.data.interaction import Interaction
from recbole.utils import get_model, init_seed
from IPython.display import FileLink

# Device auto-detect: SASRec inference is a tiny forward pass, so CPU is fine. The checkpoint was
# trained on GPU, so we override the saved device. The torch.load shim handles BOTH (a) PyTorch
# >=2.6's weights_only=True default (can't unpickle the RecBole config) and (b) mapping CUDA-saved
# storages onto this device — and it stays active through create_dataset/data_preparation, which
# torch.load the cached dataset/dataloaders too.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
_orig_torch_load = torch.load
torch.load = functools.partial(torch.load, weights_only=False, map_location=DEVICE)
try:
    ckpt = torch.load(find_data(f'{MODEL}.pth'))
    cfg = ckpt['config']
    cfg['device'] = torch.device(DEVICE)          # override the trained ('cuda') device
    init_seed(cfg['seed'], cfg['reproducibility'])
    srv_dataset = create_dataset(cfg)
    train_data, _, _ = data_preparation(cfg, srv_dataset)
    srv_model = get_model(cfg['model'])(cfg, train_data.dataset).to(DEVICE)
    srv_model.load_state_dict(ckpt['state_dict'])
    if ckpt.get('other_parameter'):
        srv_model.load_other_parameter(ckpt['other_parameter'])
finally:
    torch.load = _orig_torch_load
srv_model.eval()
print(f'loaded {MODEL} on {DEVICE}')

tok2id = srv_dataset.field2token_id[srv_dataset.iid_field]            # book_id (str) -> internal id
id2tok = [str(t) for t in srv_dataset.field2id_token[srv_dataset.iid_field]]  # internal id -> book_id
mapping = {'iid_field': srv_dataset.iid_field, 'item_id_token': id2tok,
           'item_num': len(id2tok), 'max_seq_len': int(cfg['MAX_ITEM_LIST_LENGTH'])}
with open(f'artifacts/{MODEL}_item_map.json', 'w') as fh:
    json.dump(mapping, fh)
print(f"loaded {MODEL}; exported {len(id2tok):,} item ids -> artifacts/{MODEL}_item_map.json")
display(FileLink(f'artifacts/{MODEL}_item_map.json'))

In [ ]:
SEQ = cfg['ITEM_ID_FIELD'] + cfg['LIST_SUFFIX']   # e.g. 'item_id_list'
LEN = cfg['ITEM_LIST_LENGTH_FIELD']               # e.g. 'item_length'
MAX, DEV = cfg['MAX_ITEM_LIST_LENGTH'], cfg['device']

@torch.no_grad()
def sasrec_recommend(liked_book_ids, k=10):
    """Top-K next items for an arbitrary *chronological* liked sequence — the reference impl for
    the package SasrecRecommender. Unknown ids dropped; already-liked excluded."""
    seq = [tok2id[str(b)] for b in liked_book_ids if str(b) in tok2id][-MAX:]
    if not seq:
        return []
    pad = [0] * (MAX - len(seq))
    inter = Interaction({SEQ: torch.tensor([seq + pad], device=DEV),
                         LEN: torch.tensor([len(seq)], device=DEV)})
    scores = srv_model.full_sort_predict(inter).reshape(-1)   # over every internal item id
    scores[seq] = -float('inf')                               # exclude already-liked
    return [id2tok[i] for i in torch.topk(scores, k).indices.tolist()]

example_liked = id2tok[1:6]      # a few real book_ids from the trained vocabulary (skip [PAD]=0)
print('liked  :', example_liked)
print('next-10:', sasrec_recommend(example_liked, 10))